In [3]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from libpysal.weights import KNN
from esda import Moran
import numpy as np

# -------------------------------------------------------
# 1. Load and prepare data
# -------------------------------------------------------
fastfood = gpd.read_file("Data/fastfood_location_name_amenity.gpkg")
crashes = pd.read_csv("Data/CrashStatisticsFranklinCounty.csv")

# drop missing coords
crashes = crashes.dropna(subset=["Latitude", "Longitude"])

# create GeoDataFrames
gdf_crashes = gpd.GeoDataFrame(
    crashes,
    geometry=gpd.points_from_xy(crashes["Longitude"], crashes["Latitude"]),
    crs="EPSG:4326"
)
gdf_fastfood = fastfood.copy()

# ensure CRS match
if gdf_fastfood.crs.to_epsg() != 4326:
    gdf_fastfood = gdf_fastfood.to_crs(epsg=4326)

# -------------------------------------------------------
# 2. Compute distance to nearest fast-food outlet
# -------------------------------------------------------
# for computational efficiency, project to meters
gdf_crashes_m = gdf_crashes.to_crs(epsg=3857)
gdf_fastfood_m = gdf_fastfood.to_crs(epsg=3857)

# compute nearest fast-food distance for each crash
nearest_dist = []
for crash_point in gdf_crashes_m.geometry:
    dist = gdf_fastfood_m.distance(crash_point).min()
    nearest_dist.append(dist)

gdf_crashes_m["dist_to_fastfood_m"] = nearest_dist

# -------------------------------------------------------
# 3. Build spatial weights (neighbors)
# -------------------------------------------------------
# use KNN = 8 nearest crashes as neighborhood
w = KNN.from_dataframe(gdf_crashes_m, k=8)
w.transform = 'R'

# -------------------------------------------------------
# 4. Compute Moran’s I
# -------------------------------------------------------
# variable of interest: inverse distance (closer = higher value)
gdf_crashes_m["fastfood_proximity"] = 1 / (gdf_crashes_m["dist_to_fastfood_m"] + 1)

moran = Moran(gdf_crashes_m["fastfood_proximity"], w)

# -------------------------------------------------------
# 5. Print and interpret results
# -------------------------------------------------------
print("Moran’s I statistic:", moran.I)
print("Expected I (under null):", moran.EI)
print("Z-score:", moran.z)
print("p-value:", moran.p_sim)

if moran.p_sim < 0.05:
    print("✅ Significant spatial autocorrelation detected.")
else:
    print("❌ No significant spatial autocorrelation detected.")


C:\Users\raab.75\AppData\Local\Temp\ipykernel_31036\615560613.py:12: DtypeWarning: Columns (23,31) have mixed types. Specify dtype option on import or set low_memory=False.
  crashes = pd.read_csv("Data/CrashStatisticsFranklinCounty.csv")
c:\Users\raab.75\AppData\Local\miniconda3\envs\erdos_ds_environment\Lib\site-packages\libpysal\weights\distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 29 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


Moran’s I statistic: 0.4922646996895794
Expected I (under null): -8.70852564660803e-06
Z-score: [-0.02013265 -0.11814759 -0.11870933 ...  0.07910131 -0.06720588
 -0.11196529]
p-value: 0.001
✅ Significant spatial autocorrelation detected.
